In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [ ]:
config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성
# 현재 세션 id가 abc2

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 김기원강사야.")],
    config=config,
)

print(response.content)

안녕하세요, 김기원 강사님! 만나서 반가워요. 어떻게 도와드릴까요?


In [5]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 김기원 강사님이세요. 맞나요?


In [ ]:
config = {"configurable": {"session_id": "abc3"}}
# 현재 세션 id를 abc2 -> abc3으로 변경
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

'죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 기억할게요!'

In [7]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

response.content

'아까 당신은 "안녕? 난 김기원 강사야."라고 말씀하셨고, 제가 당신의 이름을 확인했습니다. 그 외에 특별한 대화는 없었습니다. 더 이야기해보고 싶은 내용이 있으신가요?'

In [ ]:
config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="|")
# 여기서 |로 토큰 값을 확인 해보자.

# |정보|가| 부족|하지만| "|김|기|원|"|이라는| 이름|으로| 미|루|어|보|아| 한국|에서| 오|셨|을| 가능|성이| 높|습니다|.| 맞|나요|?| 
# |한국|은| 풍|부|한| 역사|와| 문|화를| 가지고| 있습니다|.| 예|를| 들어|:
# |1|.| **|전|통| 음식|**|:| 김|치|,| 불|고|기|,| 비|빔|밥| 등| 한국|음|식|이| 세계|적으로| 유명|합니다|.| 특히| 김|치는| 건강|에| 좋|고| 다양한| 재|료|로| 만들어|져| 있습니다|.
# |2|.| **|명|절|과| 기|념|일|**|:| 설|날|(|한국|의| 새|해|)|과| 추|석|(|가|을|의| 수|확|을| 기|념|하는| 명|절|)은| 가족|들이| 모|여|서| 서로|의| 안|부|를| 묻|고|,| 전|통| 음|식을| 나|누|는| 중요한| 날|입니다|.
# |3|.| **|한|복|**|:| 한국|의| 전|통| 의|상|으로|,| 결|혼|식|이나| 명|절| 같은| 특별|한| 날|에| 입|습니다|.| 화|려|하고| 다양|하게| 디자인|되어| 있습니다|.
# |4|.| **|K|-P|op|과| 드|라마|**|:| 최근| 몇| 년| 간| K|-P|op|과| 한국| 드|라마|가| 글로벌|한| 인|기를| 얻|으|면서| 한국| 문화|도| 많은| 나라|에서| 사랑|받|고| 있습니다|.
# |더| 궁|금|한| 내용|이| 있|으면| 말씀|해| 주세요|!||||

|정보|가| 부족|하지만| "|김|기|원|"|이라는| 이름|으로| 미|루|어|보|아| 한국|에서| 오|셨|을| 가능|성이| 높|습니다|.| 맞|나요|?| 

|한국|은| 풍|부|한| 역사|와| 문|화를| 가지고| 있습니다|.| 예|를| 들어|:

|1|.| **|전|통| 음식|**|:| 김|치|,| 불|고|기|,| 비|빔|밥| 등| 한국|음|식|이| 세계|적으로| 유명|합니다|.| 특히| 김|치는| 건강|에| 좋|고| 다양한| 재|료|로| 만들어|져| 있습니다|.

|2|.| **|명|절|과| 기|념|일|**|:| 설|날|(|한국|의| 새|해|)|과| 추|석|(|가|을|의| 수|확|을| 기|념|하는| 명|절|)은| 가족|들이| 모|여|서| 서로|의| 안|부|를| 묻|고|,| 전|통| 음|식을| 나|누|는| 중요한| 날|입니다|.

|3|.| **|한|복|**|:| 한국|의| 전|통| 의|상|으로|,| 결|혼|식|이나| 명|절| 같은| 특별|한| 날|에| 입|습니다|.| 화|려|하고| 다양|하게| 디자인|되어| 있습니다|.

|4|.| **|K|-P|op|과| 드|라마|**|:| 최근| 몇| 년| 간| K|-P|op|과| 한국| 드|라마|가| 글로벌|한| 인|기를| 얻|으|면서| 한국| 문화|도| 많은| 나라|에서| 사랑|받|고| 있습니다|.

|더| 궁|금|한| 내용|이| 있|으면| 말씀|해| 주세요|!||||